# Notebook to clean traffic stop data

In [ ]:
# install packages from requirements file
%pip install -r "requirements.txt"

# import necessary libraries and functions
from bias_dp_functions import *
import pandas as pd
import geopandas as gpd
import numpy as np
from astral import LocationInfo
from astral.sun import sun
from datetime import date, timedelta
from zipfile import ZipFile

### Hardcoded Values
- Edit hardcoded values here if changing the pipeline to include new features or data. 
- If reproducing same results, leave following chunk as is.

In [13]:
### Hardcoded Values

# set counties of interest
counties_of_interest = ['San Francisco']
timezone = "America/Los_Angeles"

# set dates of interest
analysis_start_date = '1-01-2010'
analysis_end_date = '06-01-2016'

# set name of date, reason for stop column in stop data
date_col = 'date'
time_col = 'time'
col_to_remap = 'reason_for_stop'

# variables of interest
var = ['date', 'time',
 'lat', 'lng',
 'subject_age', 'subject_race', 'subject_sex',
 'arrest_made', 'citation_issued', 'warning_issued',
 'reason_for_stop',
 'search_conducted', 'search_vehicle']
district_features = ['district', 'geometry']
census_features = ['pct_over75', 'pct_poc', 'pct_disab', 'epc_class', 'geometry']

# renamed features
reason_map = {
    'Moving Violation': 'moving',
    'Mechanical or Non-Moving Violation (V.C.)': 'mech_nonmoving',
    'DUI Check': 'dui',
    'Traffic Collisions': 'collision',
    'Assistance to Motorist': 'motor_assist',
    'MPC Violation': 'mpc',
    'BOLO/APB/Warrant': 'bolo'
}

final_cols = [f for f in var if f != 'reason_for_stop'] + [f for f in district_features if f != 'geometry'] + [f for f in census_features if f != 'geometry'] + list(reason_map.values())

### Load Raw Data
Load the data using the relative path directories. 
Paths do not need to be edited unless filename changes. 

In [14]:
# paths to raw data from directory
county_path = r"../data/raw_data/Bay_Area_County_Polygons.geojson"
census_path = r"../data/raw_data/communities_of_concern.geojson"
district_path = r"../data/raw_data/police_districts.geojson"
stop_path = r"../data/raw_data/sf_police_stops_raw.csv.zip"

# load data
county_df = load_data(county_path)
census_df = load_data(census_path)
district_df = load_data(district_path)
stops_df = load_data(stop_path, geospatial = True)


Bay_Area_County_Polygons.geojson converted to dataframe.
communities_of_concern.geojson converted to dataframe.
police_districts.geojson converted to dataframe.
sf_police_stops_raw.csv.zip file unzipped.
df coerced to geopandas dataframe
sf_police_stops_raw.csv.zip geospatial data converted to geopandas dataframe.


### Filtering
- Filter by date and time specified in hardcode chunk

In [15]:
# filter traffic stop locations and dates
stops_df = filter_location(stops_df, county_df, counties_of_interest)
stops_df = filter_time(stops_df, date_col, analysis_start_date, analysis_end_date)

locations filtered
time filtered


### Feature Engineering
- Add features from supplementary datasets.
- Keep only relevant features for analysis.

In [16]:
# keep only relevant variables
stops_df = stops_df[var]

# add police district, census information
stops_df = join_features(stops_df, district_df, district_features, geopandas=True)
stops_df = join_features(stops_df, census_df, census_features, geopandas=True)

# add light condition information
stops_df = create_datetime(stops_df, timezone, date_col, time_col)
sun_df = get_sun_df(stops_df[date_col].min(), stops_df[date_col].max())
stops_df = get_light_condition(stops_df, sun_df, timezone)

# simplify features with reason mapping
stops_df = simplify_col(stops_df, reason_map, col_to_remap)

# remove/edit nan values 
stops_df['epc_class'] = stops_df['epc_class'].replace({'NA':'Non-EPC'})
stops_df = stops_df.dropna()

df coerced to geopandas dataframe
new features added
df coerced to geopandas dataframe
new features added
datetime column initialized
sun status data pulled
added light conditions to df
features re-mapped


### Export Clean Data
- Export final dataset to csv and zip file

In [17]:
# keep relevant columns
stops_df = stops_df[final_cols]

# export new cleaned dataset
csv_path = r"../data/clean_data/stops_clean.csv"
stops_df.to_csv(csv_path, index=False)

import subprocess
subprocess.run(['zip', '-j', r"../data/clean_data/stops_clean.csv.zip", r"../data/clean_data/stops_clean.csv"])

updating: stops_clean.csv
	zip warning: Local Entry CRC does not match CD: stops_clean.csv
 (deflated 85%)


CompletedProcess(args=['zip', '-j', '../data/clean_data/stops_clean.csv.zip', '../data/clean_data/stops_clean.csv'], returncode=0)